# 01 - Ingesta Layer 0 SECOP Big Data

Este notebook documenta el proceso de construcción de la **capa Layer 0** del proyecto SECOP Big Data.

La capa Layer 0 tiene como objetivo **descargar, almacenar y validar los datos crudos** provenientes de las APIs públicas de SECOP y otras fuentes de apoyo, sin aplicar todavía una limpieza profunda. En esta fase se prioriza la trazabilidad: saber de dónde vienen los datos, bajo qué ventana temporal fueron consultados, cuántos registros se descargaron, en qué ruta quedaron guardados y bajo qué identificador de ejecución se almacenaron.

El flujo general del notebook es:

```text
1. Crear el Volume en Databricks.
2. Crear la estructura de carpetas.
3. Probar la conexión con la API.
4. Descargar un lote pequeño de prueba.
5. Validar lectura con Spark.
6. Convertir datos crudos a CSV y Parquet.
7. Incorporar múltiples fuentes.
8. Escalar la descarga por lotes.
9. Ajustar la regla final de consulta.
10. Descargar datos desde 2025-01-01 hasta la fecha de descarga.
11. Buscar y validar los datos realmente descargados.
12. Reconstruir Parquet desde JSONL validado.
```

## Paso 1. Crear el Volume en Unity Catalog

En este bloque se crea el Volume llamado `tallerspark` dentro del catálogo `workspace` y el esquema `default`.

Un **Volume** en Databricks funciona como una zona administrada para almacenar archivos. En este proyecto se utiliza para guardar los datos descargados desde SECOP, separados del repositorio Git. Esto es importante porque GitHub debe guardar solamente código y notebooks, no archivos masivos de datos.

La ruta que se crea es:

```text
/Volumes/workspace/default/tallerspark/
```

A partir de esta ruta se construye la estructura del proyecto.


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.tallerspark;

## Paso 2. Crear la estructura inicial de carpetas del proyecto

En este bloque se define la ruta base del proyecto:

```text
/Volumes/workspace/default/tallerspark/secop
```

Luego se crean carpetas para organizar los archivos de la ingesta:

```text
raw_jsonl/   → datos crudos descargados de la API en formato JSONL
raw_csv/     → copia en CSV para visualización o evidencia académica
parquet/     → copia optimizada para lectura con Spark
manifest/    → archivos de trazabilidad de las descargas
logs/        → carpeta para registros operativos
```

Esta organización permite separar claramente los datos crudos, los datos optimizados y los archivos de control.

In [0]:
BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

folders = [
    f"{BASE_PATH}/raw_jsonl/secop_ii_contratos/year=2025",
    f"{BASE_PATH}/raw_csv/secop_ii_contratos/year=2025",
    f"{BASE_PATH}/parquet/secop_ii_contratos/year=2025",
    f"{BASE_PATH}/manifest",
    f"{BASE_PATH}/logs"
]

for folder in folders:
    dbutils.fs.mkdirs(folder)
    print(f"Created: {folder}")

## Paso 3. Verificar que la estructura fue creada

Esta celda lista el contenido de la ruta base del proyecto.

La validación esperada es que aparezcan carpetas como:

```text
raw_jsonl/
raw_csv/
parquet/
manifest/
logs/
```

Si estas carpetas aparecen, significa que el entorno de almacenamiento está listo para recibir la descarga.

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/tallerspark/secop"))

## Paso 4. Configurar la primera fuente de datos: SECOP II Contratos

Este bloque configura la primera prueba de descarga usando la fuente principal:

```text
SECOP II - Contratos electrónicos
```

Se definen variables clave:

```text
YEAR        → año inicial de análisis
START_DATE  → fecha inicial
END_DATE    → fecha final usada en la primera prueba
BASE_URL    → endpoint de la API
DATE_COLUMN → columna de fecha usada para filtrar
```

En esta etapa todavía se usa una configuración inicial cerrada al año 2025. Más adelante el notebook cambia la regla para ajustarse al requerimiento final: desde `2025-01-01` hasta la fecha real de descarga.


In [0]:
import requests
import json
import pandas as pd
from datetime import datetime, timezone
import time

YEAR = 2025

START_DATE = "2025-01-01T00:00:00"
END_DATE = "2026-01-01T00:00:00"

SOURCE_NAME = "secop_ii_contratos"
BASE_URL = "https://www.datos.gov.co/resource/jbjy-vk9h.json"
DATE_COLUMN = "fecha_de_firma"

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

RAW_JSONL_PATH = f"{BASE_PATH}/raw_jsonl/{SOURCE_NAME}/year={YEAR}"
RAW_CSV_PATH = f"{BASE_PATH}/raw_csv/{SOURCE_NAME}/year={YEAR}"
PARQUET_PATH = f"{BASE_PATH}/parquet/{SOURCE_NAME}/year={YEAR}"
MANIFEST_PATH = f"{BASE_PATH}/manifest"

BATCH_SIZE = 1000
MAX_BATCHES = 1

## Paso 5. Probar la conexión con la API

Antes de descargar miles de registros, este bloque hace una prueba mínima con solo 5 registros.

Esto permite validar tres cosas:

```text
1. Que la API responde correctamente.
2. Que el filtro de fecha no genera error.
3. Que los datos se pueden convertir a un DataFrame para inspección.
```

Si esta celda muestra 5 registros, la conexión con SECOP funciona correctamente.


In [0]:
params_test = {
    "$limit": 5,
    "$offset": 0,
    "$where": f"{DATE_COLUMN} >= '{START_DATE}' AND {DATE_COLUMN} < '{END_DATE}'"
}

response = requests.get(BASE_URL, params=params_test, timeout=60)
response.raise_for_status()

sample_data = response.json()

print("Records downloaded:", len(sample_data))

df_sample = pd.DataFrame(sample_data)
display(df_sample)

## Paso 6. Crear función de descarga por lote

Aquí se define una función llamada `fetch_secop_batch`.

Esta función recibe:

```text
limit  → cantidad máxima de registros a descargar
offset → posición inicial del lote
```

La API de Socrata permite paginar resultados usando `$limit` y `$offset`. Esto es fundamental para un enfoque Big Data, porque evita intentar descargar todos los datos de una sola vez.

La función también incluye reintentos. Si la API falla temporalmente, el proceso espera unos segundos y vuelve a intentar.

In [0]:
def fetch_secop_batch(limit, offset, retries=3, sleep_seconds=5):
    params = {
        "$limit": limit,
        "$offset": offset,
        "$where": f"{DATE_COLUMN} >= '{START_DATE}' AND {DATE_COLUMN} < '{END_DATE}'",
        "$order": f"{DATE_COLUMN} ASC"
    }

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(BASE_URL, params=params, timeout=120)
            response.raise_for_status()
            return response.json()

        except Exception as error:
            print(f"Attempt {attempt} failed at offset {offset}: {error}")

            if attempt == retries:
                raise

            time.sleep(sleep_seconds)

## Paso 7. Descargar el primer lote de prueba

Este bloque descarga un primer lote de prueba de 1.000 registros.

La lógica es:

```text
BATCH_SIZE = 1000
MAX_BATCHES = 1
```

Esto significa que solo se descarga un lote pequeño para validar el proceso completo antes de escalar.

Los datos se guardan como archivo `.jsonl`, donde cada línea contiene un registro JSON. Este formato es adecuado para Layer 0 porque conserva la respuesta de la API casi sin transformaciones.

In [0]:
import requests
import json
import time
from datetime import datetime, timezone

YEAR = 2025

START_DATE = "2025-01-01T00:00:00"
END_DATE = "2026-01-01T00:00:00"

SOURCE_NAME = "secop_ii_contratos"
BASE_URL = "https://www.datos.gov.co/resource/jbjy-vk9h.json"
DATE_COLUMN = "fecha_de_firma"

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

RAW_JSONL_PATH = f"{BASE_PATH}/raw_jsonl/{SOURCE_NAME}/year={YEAR}"
MANIFEST_PATH = f"{BASE_PATH}/manifest"

BATCH_SIZE = 1000
MAX_BATCHES = 1


def fetch_secop_batch(limit, offset, retries=3, sleep_seconds=5):
    params = {
        "$limit": limit,
        "$offset": offset,
        "$where": f"{DATE_COLUMN} >= '{START_DATE}' AND {DATE_COLUMN} < '{END_DATE}'",
        "$order": f"{DATE_COLUMN} ASC"
    }

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(BASE_URL, params=params, timeout=120)
            response.raise_for_status()
            return response.json()

        except Exception as error:
            print(f"Attempt {attempt} failed at offset {offset}: {error}")

            if attempt == retries:
                raise

            time.sleep(sleep_seconds)


manifest = []

for batch_number in range(MAX_BATCHES):
    offset = batch_number * BATCH_SIZE

    print(f"Downloading batch {batch_number} | offset={offset} | limit={BATCH_SIZE}")

    records = fetch_secop_batch(
        limit=BATCH_SIZE,
        offset=offset
    )

    print("Records received:", len(records))

    if not records:
        print("No more records available.")
        break

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

    jsonl_file = (
        f"{RAW_JSONL_PATH}/"
        f"batch_{batch_number:04d}_offset_{offset}_limit_{BATCH_SIZE}_{timestamp}.jsonl"
    )

    # Correct for Databricks Unity Catalog Volumes
    with open(jsonl_file, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    manifest.append({
        "source": SOURCE_NAME,
        "year": YEAR,
        "batch_number": batch_number,
        "offset": offset,
        "limit": BATCH_SIZE,
        "records": len(records),
        "jsonl_file": jsonl_file,
        "download_utc": datetime.now(timezone.utc).isoformat()
    })

    print("Saved JSONL:", jsonl_file)

## Paso 8. Leer el archivo JSONL con Spark

Después de guardar los datos crudos, se leen con Spark desde la carpeta `raw_jsonl`.

Esta validación confirma que:

```text
1. El archivo quedó guardado correctamente.
2. Spark puede interpretar el JSONL.
3. El conteo de registros coincide con lo descargado.
4. Las columnas fueron inferidas correctamente.
```

En este punto todavía no se hace limpieza. Solo se valida la lectura del dato crudo.

In [0]:
df_raw = spark.read.json(RAW_JSONL_PATH)

print("Total records:", df_raw.count())
print("Total columns:", len(df_raw.columns))

display(df_raw.limit(10))

## Paso 9. Revisar el esquema del dato crudo

`printSchema()` muestra la estructura que Spark detectó automáticamente.

Esta revisión permite identificar:

```text
- nombres de columnas
- tipos de datos inferidos
- columnas anidadas
- campos que requieren transformación posterior
```

Este paso es importante porque algunas columnas vienen como texto, otras como estructuras anidadas y otras pueden requerir conversión en Silver.

In [0]:
df_raw.printSchema()

## Paso 10. Generar copia CSV de evidencia

Este bloque convierte el DataFrame crudo a CSV.

El CSV no es el formato más eficiente para Big Data, pero es útil para:

```text
- revisión visual
- evidencia académica
- descarga manual
- inspección rápida
```

En este caso también se aplana la columna `urlproceso`, porque las columnas tipo estructura no se pueden guardar directamente en CSV.

In [0]:
RAW_CSV_PATH = "/Volumes/workspace/default/tallerspark/secop/raw_csv/secop_ii_contratos/year=2025/batch_0000_csv"

# Flatten struct columns for CSV export
df_flat = df_raw.withColumn("urlproceso_url", df_raw.urlproceso.url).drop("urlproceso")

(
    df_flat.write
    .mode("overwrite")
    .option("header", "true")
    .csv(RAW_CSV_PATH)
)

print("CSV saved in:", RAW_CSV_PATH)

## Paso 11. Guardar copia en Parquet

Este bloque guarda los datos en formato Parquet.

Parquet es un formato columnar, comprimido y eficiente para Spark. Es mucho más adecuado que CSV para procesamiento distribuido.

La idea es:

```text
JSONL  → conserva la respuesta cruda
Parquet → permite procesar más rápido en Spark
CSV    → sirve como evidencia o revisión manual
```

In [0]:
PARQUET_PATH = "/Volumes/workspace/default/tallerspark/secop/parquet/secop_ii_contratos/year=2025/batch_0000_parquet"

(
    df_raw.write
    .mode("overwrite")
    .parquet(PARQUET_PATH)
)

print("Parquet saved in:", PARQUET_PATH)

## Paso 12. Crear manifiesto de descarga

El manifiesto documenta la descarga realizada.

Incluye información como:

```text
fuente
año
número de lote
offset
límite
cantidad de registros
ruta JSONL
ruta CSV
ruta Parquet
fecha de creación
```

Este archivo es clave para trazabilidad, porque permite demostrar qué se descargó, cuándo se descargó y dónde quedó almacenado.

In [0]:
import json
from datetime import datetime, timezone

MANIFEST_PATH = "/Volumes/workspace/default/tallerspark/secop/manifest/manifest_secop_2025_batch_0000.json"

manifest_data = {
    "source": "secop_ii_contratos",
    "year": 2025,
    "batch_number": 0,
    "offset": 0,
    "limit": 1000,
    "records": df_raw.count(),
    "raw_jsonl_path": RAW_JSONL_PATH,
    "raw_csv_path": RAW_CSV_PATH,
    "parquet_path": PARQUET_PATH,
    "created_utc": datetime.now(timezone.utc).isoformat()
}

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest_data, f, ensure_ascii=False, indent=4)

print("Manifest saved:", MANIFEST_PATH)

## Paso 13. Verificación final de carpetas iniciales

Esta celda vuelve a listar la ruta base del proyecto.

Sirve para confirmar que las carpetas y archivos generados en las primeras pruebas existen dentro del Volume.

In [0]:
BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

display(dbutils.fs.ls(BASE_PATH))

## Paso 14. Configurar múltiples fuentes de datos

En este bloque se amplía el proyecto para trabajar con varias fuentes:

```text
secop_ii_contratos
secop_ii_adiciones
secop_ii_ejecucion
divipola_municipios
```

Cada fuente tiene:

```text
url                  → endpoint de la API
date_column          → columna usada para filtrar por fecha
apply_year_filter    → indica si aplica filtro temporal
```

DIVIPOLA no requiere filtro de año porque es una tabla de referencia territorial.

In [0]:
import os
import requests
import json
import time
from datetime import datetime, timezone

YEAR = 2025
START_DATE = "2025-01-01T00:00:00"
END_DATE = "2026-01-01T00:00:00"

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

BATCH_SIZE = 1000
MAX_BATCHES = 1

SOURCES = {
    "secop_ii_contratos": {
        "url": "https://www.datos.gov.co/resource/jbjy-vk9h.json",
        "date_column": "fecha_de_firma",
        "apply_year_filter": True
    },
    "secop_ii_adiciones": {
        "url": "https://www.datos.gov.co/resource/cb9c-h8sn.json",
        "date_column": "fecharegistro",
        "apply_year_filter": True
    },
    "secop_ii_ejecucion": {
        "url": "https://www.datos.gov.co/resource/mfmm-jqmq.json",
        "date_column": "fechacreacion",
        "apply_year_filter": True
    },
    "divipola_municipios": {
        "url": "https://www.datos.gov.co/resource/gdxc-w37w.json",
        "date_column": None,
        "apply_year_filter": False
    }
}

## Paso 15. Crear carpetas para todas las fuentes

Aquí se crea la estructura Layer 0 para cada fuente.

Para cada fuente se crean carpetas en:

```text
raw_jsonl/<fuente>/
raw_csv/<fuente>/
parquet/<fuente>/
```

Esto permite mantener separadas las descargas de contratos, adiciones, ejecución y DIVIPOLA.

In [0]:
for source_name in SOURCES.keys():
    folders = [
        f"{BASE_PATH}/raw_jsonl/{source_name}/year={YEAR}",
        f"{BASE_PATH}/raw_csv/{source_name}/year={YEAR}",
        f"{BASE_PATH}/parquet/{source_name}/year={YEAR}"
    ]

    for folder in folders:
        os.makedirs(folder, exist_ok=True)
        print(f"Folder ready: {folder}")

os.makedirs(f"{BASE_PATH}/manifest", exist_ok=True)
os.makedirs(f"{BASE_PATH}/logs", exist_ok=True)

## Paso 16. Funciones generales para descargar cualquier fuente

Este bloque generaliza la lógica de descarga.

Las funciones principales son:

```text
build_params()             → construye los parámetros de la API
fetch_batch()              → descarga un lote con reintentos
download_source_layer0()   → descarga, guarda JSONL y crea manifiesto
```

Este es un punto importante del proyecto porque ya no se trabaja con código específico solo para contratos. La lógica queda parametrizada y reutilizable para varias fuentes.


In [0]:
def build_params(source_config, limit, offset):
    params = {
        "$limit": limit,
        "$offset": offset
    }

    if source_config["apply_year_filter"]:
        date_column = source_config["date_column"]
        params["$where"] = f"{date_column} >= '{START_DATE}' AND {date_column} < '{END_DATE}'"
        params["$order"] = f"{date_column} ASC"

    return params


def fetch_batch(source_config, limit, offset, retries=3, sleep_seconds=5):
    params = build_params(source_config, limit, offset)

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(source_config["url"], params=params, timeout=120)
            response.raise_for_status()
            return response.json()

        except Exception as error:
            print(f"Attempt {attempt} failed at offset {offset}: {error}")

            if attempt == retries:
                raise

            time.sleep(sleep_seconds)


def download_source_layer0(source_name, batch_size=1000, max_batches=1):
    source_config = SOURCES[source_name]

    raw_jsonl_path = f"{BASE_PATH}/raw_jsonl/{source_name}/year={YEAR}"
    manifest = []

    for batch_number in range(max_batches):
        offset = batch_number * batch_size

        print(f"Downloading source={source_name} | batch={batch_number} | offset={offset} | limit={batch_size}")

        records = fetch_batch(source_config, batch_size, offset)

        print(f"Records received: {len(records)}")

        if not records:
            print("No more records available.")
            break

        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

        jsonl_file = (
            f"{raw_jsonl_path}/"
            f"batch_{batch_number:04d}_offset_{offset}_limit_{batch_size}_{timestamp}.jsonl"
        )

        with open(jsonl_file, "w", encoding="utf-8") as f:
            for record in records:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        manifest.append({
            "source": source_name,
            "year": YEAR,
            "batch_number": batch_number,
            "offset": offset,
            "limit": batch_size,
            "records": len(records),
            "jsonl_file": jsonl_file,
            "download_utc": datetime.now(timezone.utc).isoformat()
        })

        print(f"Saved JSONL: {jsonl_file}")

    manifest_file = f"{BASE_PATH}/manifest/manifest_{source_name}_{YEAR}.json"

    with open(manifest_file, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=4)

    print(f"Manifest saved: {manifest_file}")

    return manifest

## Paso 17. Descargar fuentes adicionales

En esta etapa se descargan las fuentes complementarias:

```text
secop_ii_adiciones
secop_ii_ejecucion
divipola_municipios
```

La fuente `secop_ii_contratos` ya había sido probada antes. Aquí se valida que el patrón de ingesta sirve para múltiples datasets.

In [0]:
sources_to_download = [
    "secop_ii_adiciones",
    "secop_ii_ejecucion",
    "divipola_municipios"
]

all_manifests = {}

for source_name in sources_to_download:
    manifest = download_source_layer0(
        source_name=source_name,
        batch_size=1000,
        max_batches=1
    )

    all_manifests[source_name] = manifest

## Paso 18. Validar archivos JSONL descargados por fuente

Este bloque lista los archivos crudos guardados en `raw_jsonl`.

La validación esperada es encontrar archivos `.jsonl` para cada fuente.

Si una fuente no muestra archivos, significa que la descarga falló o que la ruta consultada no corresponde al lugar donde se guardaron los datos.

In [0]:
for source_name in SOURCES.keys():
    path = f"{BASE_PATH}/raw_jsonl/{source_name}/year={YEAR}"
    print("\nSOURCE:", source_name)
    display(dbutils.fs.ls(path))

## Paso 19. Convertir todas las fuentes a CSV y Parquet

Después de descargar los JSONL, este bloque convierte todas las fuentes a:

```text
CSV     → revisión manual y evidencia
Parquet → procesamiento eficiente con Spark
```

También incluye una lógica defensiva para tratar columnas anidadas antes de escribir CSV.

Este paso prepara los datos para futuras capas, especialmente Bronze y Silver.

In [0]:
for source_name in SOURCES.keys():
    raw_jsonl_path = f"{BASE_PATH}/raw_jsonl/{source_name}/year={YEAR}"
    raw_csv_path = f"{BASE_PATH}/raw_csv/{source_name}/year={YEAR}/batch_0000_csv"
    parquet_path = f"{BASE_PATH}/parquet/{source_name}/year={YEAR}/batch_0000_parquet"

    print(f"Processing source: {source_name}")

    df = spark.read.json(raw_jsonl_path)

    row_count = df.count()

    print(f"Rows: {row_count} | Columns: {len(df.columns)}")

    # Compute schema once after loading
    schema_fields = df.schema.fields

    # Flatten struct columns for CSV export
    # Build column expressions all at once to avoid nested plans
    columns_to_select = []
    for field in schema_fields:
        if str(field.dataType).startswith("StructType"):
            # Extract nested field and flatten
            if field.name == "urlproceso" and "url" in [f.name for f in field.dataType.fields]:
                columns_to_select.append(df[field.name].url.alias(f"{field.name}_url"))
        else:
            columns_to_select.append(df[field.name])

    df_for_csv = df.select(*columns_to_select)

    (
        df_for_csv.write
        .mode("overwrite")
        .option("header", "true")
        .csv(raw_csv_path)
    )

    (
        df.write
        .mode("overwrite")
        .parquet(parquet_path)
    )

    print(f"CSV saved: {raw_csv_path}")
    print(f"Parquet saved: {parquet_path}")

## Paso 20. Preparar escalamiento a 100.000 registros

Aquí se cambia la estrategia de prueba pequeña a una descarga mayor.

La configuración:

```text
BATCH_SIZE = 50000
MAX_BATCHES = 2
```

equivale a:

```text
50.000 registros × 2 lotes = 100.000 registros
```

Esta etapa demuestra escalabilidad progresiva sin descargar todavía todo el universo disponible.

In [0]:
BATCH_SIZE = 50000
MAX_BATCHES = 2

sources_to_escalate = [
    "secop_ii_contratos",
    "secop_ii_adiciones",
    "secop_ii_ejecucion"
]

## Paso 21. Crear identificador de ejecución

El `RUN_ID` identifica una ejecución específica.

Esto permite separar descargas diferentes, por ejemplo:

```text
prueba de 1.000 registros
prueba de 100.000 registros
descarga completa
```

Usar `run_id` evita mezclar resultados y permite trazabilidad.

In [0]:
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

print("Current run ID:", RUN_ID)

## Paso 22. Descargar hasta 100.000 registros por fuente

Este bloque implementa una descarga controlada de 100.000 registros para las fuentes principales.

La lógica usa:

```text
limit  = 50000
offset = 0, 50000
```

Cada lote se guarda en una carpeta con `run_id`.

Esto permite probar un volumen mayor antes de pasar a la descarga completa.

In [0]:
import os
import requests
import json
import time
from datetime import datetime, timezone

YEAR = 2025
START_DATE = "2025-01-01T00:00:00"
END_DATE = "2026-01-01T00:00:00"

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

BATCH_SIZE = 50000
MAX_BATCHES = 2

SOURCES = {
    "secop_ii_contratos": {
        "url": "https://www.datos.gov.co/resource/jbjy-vk9h.json",
        "date_column": "fecha_de_firma",
        "apply_year_filter": True
    },
    "secop_ii_adiciones": {
        "url": "https://www.datos.gov.co/resource/cb9c-h8sn.json",
        "date_column": "fecharegistro",
        "apply_year_filter": True
    },
    "secop_ii_ejecucion": {
        "url": "https://www.datos.gov.co/resource/mfmm-jqmq.json",
        "date_column": "fechacreacion",
        "apply_year_filter": True
    }
}

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

print("Run ID:", RUN_ID)


def build_params(source_config, limit, offset):
    params = {
        "$limit": limit,
        "$offset": offset
    }

    if source_config["apply_year_filter"]:
        date_column = source_config["date_column"]
        params["$where"] = f"{date_column} >= '{START_DATE}' AND {date_column} < '{END_DATE}'"
        params["$order"] = f"{date_column} ASC"

    return params


def fetch_batch(source_config, limit, offset, retries=3, sleep_seconds=10):
    params = build_params(source_config, limit, offset)

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(source_config["url"], params=params, timeout=180)
            response.raise_for_status()

            data = response.json()

            if isinstance(data, dict) and "error" in data:
                raise Exception(data)

            return data

        except Exception as error:
            print(f"Attempt {attempt} failed | offset={offset} | error={error}")

            if attempt == retries:
                raise

            time.sleep(sleep_seconds)


def download_source_100k(source_name):
    source_config = SOURCES[source_name]

    raw_jsonl_path = (
        f"{BASE_PATH}/raw_jsonl/{source_name}/year={YEAR}/run_id={RUN_ID}"
    )

    os.makedirs(raw_jsonl_path, exist_ok=True)

    manifest = []

    total_records = 0

    for batch_number in range(MAX_BATCHES):
        offset = batch_number * BATCH_SIZE

        print("=" * 80)
        print(f"Downloading source={source_name}")
        print(f"Batch={batch_number} | Offset={offset} | Limit={BATCH_SIZE}")

        records = fetch_batch(
            source_config=source_config,
            limit=BATCH_SIZE,
            offset=offset
        )

        received = len(records)
        total_records += received

        print(f"Records received: {received}")

        if received == 0:
            print("No more records available.")
            break

        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

        jsonl_file = (
            f"{raw_jsonl_path}/"
            f"batch_{batch_number:04d}_offset_{offset}_limit_{BATCH_SIZE}_{timestamp}.jsonl"
        )

        with open(jsonl_file, "w", encoding="utf-8") as f:
            for record in records:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        manifest.append({
            "source": source_name,
            "year": YEAR,
            "run_id": RUN_ID,
            "batch_number": batch_number,
            "offset": offset,
            "limit": BATCH_SIZE,
            "records": received,
            "jsonl_file": jsonl_file,
            "download_utc": datetime.now(timezone.utc).isoformat()
        })

        print(f"Saved JSONL: {jsonl_file}")

        if received < BATCH_SIZE:
            print("Last available batch reached.")
            break

    manifest_file = f"{BASE_PATH}/manifest/manifest_{source_name}_{YEAR}_run_{RUN_ID}.json"

    with open(manifest_file, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=4)

    print("=" * 80)
    print(f"Finished source: {source_name}")
    print(f"Total records downloaded: {total_records}")
    print(f"Manifest saved: {manifest_file}")

    return manifest

## Paso 23. Ejecutar descarga escalada de 100.000 registros

Esta celda ejecuta la función de descarga para cada fuente definida en `sources_to_escalate`.

El resultado esperado es un manifiesto por fuente con el detalle de los lotes descargados.

In [0]:
all_manifests_100k = {}

for source_name in sources_to_escalate:
    all_manifests_100k[source_name] = download_source_100k(source_name)

## Paso 24. Ajustar la regla final de consulta

Aquí se implementa el cambio más importante del proyecto final.

La regla deja de ser:

```text
solo año 2025 cerrado
```

y pasa a ser:

```text
desde 2025-01-01 hasta la fecha y hora real de descarga
```

Para lograrlo, el notebook captura la hora actual en zona horaria de Bogotá y la usa como `END_DATE`.

Esto asegura que todas las fuentes usen una misma fecha de corte.

In [0]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

# ============================================================
# FINAL QUERY RULE
# From 2025-01-01 to the real download datetime
# ============================================================

DOWNLOAD_TZ = ZoneInfo("America/Bogota")

DOWNLOAD_DATETIME = datetime.now(DOWNLOAD_TZ)
DOWNLOAD_DATE = DOWNLOAD_DATETIME.strftime("%Y-%m-%d")
DOWNLOAD_TIMESTAMP = DOWNLOAD_DATETIME.strftime("%Y-%m-%dT%H:%M:%S")

START_DATE = "2025-01-01T00:00:00"
END_DATE = DOWNLOAD_TIMESTAMP

PERIOD_NAME = f"from_2025_01_01_to_{DOWNLOAD_DATE}"
RUN_ID_FULL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

print("Start date:", START_DATE)
print("Download end date:", END_DATE)
print("Period:", PERIOD_NAME)
print("Run ID:", RUN_ID_FULL)

## Paso 25. Construir parámetros con la regla final

Esta función crea el filtro final para la API:

```text
date_column >= START_DATE
AND date_column <= END_DATE
```

Este filtro se aplica solamente a las fuentes que tienen columna de fecha.

DIVIPOLA queda sin filtro temporal porque es una tabla de referencia.

In [0]:
def build_params(source_config, limit, offset):
    params = {
        "$limit": limit,
        "$offset": offset
    }

    if source_config["apply_date_filter"]:
        date_column = source_config["date_column"]

        params["$where"] = (
            f"{date_column} >= '{START_DATE}' "
            f"AND {date_column} <= '{END_DATE}'"
        )

        params["$order"] = f"{date_column} ASC"

    return params

## Paso 26. Configurar fuentes con filtro dinámico

En esta celda se redefine la configuración de fuentes usando `apply_date_filter`.

La diferencia frente a la versión anterior es que ahora la lógica ya no habla de `year_filter`, sino de un filtro dinámico basado en la fecha real de descarga.

Esto deja el notebook alineado con la regla final del proyecto.

In [0]:
SOURCES_FULL = {
    "secop_ii_contratos": {
        "url": "https://www.datos.gov.co/resource/jbjy-vk9h.json",
        "date_column": "fecha_de_firma",
        "apply_date_filter": True
    },
    "secop_ii_adiciones": {
        "url": "https://www.datos.gov.co/resource/cb9c-h8sn.json",
        "date_column": "fecharegistro",
        "apply_date_filter": True
    },
    "secop_ii_ejecucion": {
        "url": "https://www.datos.gov.co/resource/mfmm-jqmq.json",
        "date_column": "fechacreacion",
        "apply_date_filter": True
    },
    "divipola_municipios": {
        "url": "https://www.datos.gov.co/resource/gdxc-w37w.json",
        "date_column": None,
        "apply_date_filter": False
    }
}

## Paso 27. Construir ruta de salida con período y run_id

Esta celda muestra la estructura final de almacenamiento:

```text
raw_jsonl/<fuente>/period=<PERIOD_NAME>/run_id=<RUN_ID_FULL>
```

Esta estructura es más clara que usar solo `year=2025`, porque el período ahora llega hasta la fecha de descarga.

In [0]:
raw_jsonl_path = (
    f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"
)

## Paso 28. Función de descarga completa por fuente

Esta función descarga todos los registros disponibles para una fuente dentro de la ventana temporal definida.

La función usa un ciclo `while True` y se detiene cuando la API devuelve menos registros que el tamaño del lote.

La lógica es:

```text
Batch 0 → offset 0
Batch 1 → offset 50000
Batch 2 → offset 100000
...
Se detiene cuando records_received < BATCH_SIZE
```

Esto permite descargar todos los datos disponibles sin conocer previamente el total de registros.

In [0]:
def download_all_source(source_name):
    source_config = SOURCES_FULL[source_name]

    raw_jsonl_path = (
        f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"
    )

    os.makedirs(raw_jsonl_path, exist_ok=True)

    manifest = []
    total_records = 0
    batch_number = 0

    while True:
        offset = batch_number * BATCH_SIZE

        print("=" * 100)
        print(f"Source: {source_name}")
        print(f"Batch: {batch_number}")
        print(f"Offset: {offset}")
        print(f"Limit: {BATCH_SIZE}")
        print(f"Query window: {START_DATE} to {END_DATE}")

        records = fetch_batch(
            source_config=source_config,
            limit=BATCH_SIZE,
            offset=offset
        )

        received = len(records)
        total_records += received

        print(f"Records received: {received}")

        if received == 0:
            print("No more records available.")
            break

        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

        jsonl_file = (
            f"{raw_jsonl_path}/"
            f"batch_{batch_number:04d}_"
            f"offset_{offset}_"
            f"limit_{BATCH_SIZE}_"
            f"{timestamp}.jsonl"
        )

        with open(jsonl_file, "w", encoding="utf-8") as f:
            for record in records:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        manifest.append({
            "source": source_name,
            "period": PERIOD_NAME,
            "start_date": START_DATE,
            "download_end_date": END_DATE,
            "run_id": RUN_ID_FULL,
            "batch_number": batch_number,
            "offset": offset,
            "limit": BATCH_SIZE,
            "records": received,
            "jsonl_file": jsonl_file,
            "download_utc": datetime.now(timezone.utc).isoformat()
        })

        print(f"Saved JSONL: {jsonl_file}")

        if received < BATCH_SIZE:
            print("Last batch reached.")
            break

        batch_number += 1
        time.sleep(3)

    manifest_file = (
        f"{BASE_PATH}/manifest/"
        f"manifest_{source_name}_{PERIOD_NAME}_run_{RUN_ID_FULL}.json"
    )

    with open(manifest_file, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=4)

    print("=" * 100)
    print(f"Finished source: {source_name}")
    print(f"Total records downloaded: {total_records}")
    print(f"Manifest saved: {manifest_file}")

    return {
        "source": source_name,
        "period": PERIOD_NAME,
        "start_date": START_DATE,
        "download_end_date": END_DATE,
        "total_records": total_records,
        "batches": len(manifest),
        "manifest_file": manifest_file,
        "run_id": RUN_ID_FULL
    }

## Paso 29. Definir fuentes para la descarga final

Aquí se define la lista de fuentes que participarán en la descarga completa:

```text
secop_ii_contratos
secop_ii_adiciones
secop_ii_ejecucion
divipola_municipios
```

Esta lista se usa en las celdas posteriores para validar, convertir y procesar todas las fuentes de forma uniforme.

In [0]:
# ============================================================
# FUENTES A PROCESAR EN LA DESCARGA FINAL
# ============================================================

sources_to_download_full = [
    "secop_ii_contratos",
    "secop_ii_adiciones",
    "secop_ii_ejecucion",
    "divipola_municipios"
]

print("Sources ready:")
for source in sources_to_download_full:
    print("-", source)

## Paso 30. Confirmar variables activas

Esta celda imprime las variables clave del proceso:

```text
BASE_PATH
PERIOD_NAME
RUN_ID_FULL
```

Es una validación operativa importante porque si el `run_id` cambia accidentalmente, el notebook puede buscar datos en una carpeta equivocada.

In [0]:
print("BASE_PATH:", BASE_PATH)
print("PERIOD_NAME:", PERIOD_NAME)
print("RUN_ID_FULL:", RUN_ID_FULL)

## Paso 31. Reconfigurar filtro dinámico si es necesario

Esta celda reconstruye la configuración de fechas y rutas.

Debe usarse con cuidado: si se ejecuta nuevamente, genera un nuevo `RUN_ID_FULL`.

Si ya existe una descarga completada, conviene fijar manualmente el `RUN_ID_FULL` correcto para no perder la referencia a la carpeta real donde están los datos.

In [0]:
import os
import requests
import json
import time
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

DOWNLOAD_TZ = ZoneInfo("America/Bogota")

DOWNLOAD_DATETIME = datetime.now(DOWNLOAD_TZ)
DOWNLOAD_DATE = DOWNLOAD_DATETIME.strftime("%Y-%m-%d")
DOWNLOAD_TIMESTAMP = DOWNLOAD_DATETIME.strftime("%Y-%m-%dT%H:%M:%S")

START_DATE = "2025-01-01T00:00:00"
END_DATE = DOWNLOAD_TIMESTAMP

PERIOD_NAME = f"from_2025_01_01_to_{DOWNLOAD_DATE}"
RUN_ID_FULL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"
BATCH_SIZE = 50000

print("Start date:", START_DATE)
print("End date:", END_DATE)
print("Period:", PERIOD_NAME)
print("Run ID:", RUN_ID_FULL)

## Paso 32. Descarga completa Layer 0 en una sola celda

Este bloque contiene una versión consolidada del proceso completo:

```text
1. Crear Volume.
2. Definir fechas dinámicas.
3. Definir fuentes.
4. Crear carpetas.
5. Descargar todos los lotes.
6. Guardar JSONL.
7. Crear manifiestos.
8. Validar datos.
9. Convertir a Parquet y CSV.
10. Crear resumen global.
```

Es una versión integral para ejecutar el flujo completo de Layer 0.

Sin embargo, en entornos gratuitos o limitados puede ser pesado porque combina descarga, validación y conversión en una misma ejecución.

In [0]:
# ============================================================
# SECOP BIG DATA - LAYER 0 FULL DOWNLOAD
# Final rule:
# From 2025-01-01 to the real download datetime
# ============================================================

import os
import json
import time
import requests
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

# ============================================================
# 1. CREATE / ENSURE UNITY CATALOG VOLUME
# ============================================================

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.tallerspark")

# ============================================================
# 2. GENERAL CONFIGURATION
# ============================================================

DOWNLOAD_TZ = ZoneInfo("America/Bogota")

DOWNLOAD_DATETIME = datetime.now(DOWNLOAD_TZ)
DOWNLOAD_DATE = DOWNLOAD_DATETIME.strftime("%Y-%m-%d")
DOWNLOAD_TIMESTAMP = DOWNLOAD_DATETIME.strftime("%Y-%m-%dT%H:%M:%S")

START_DATE = "2025-01-01T00:00:00"
END_DATE = DOWNLOAD_TIMESTAMP

PERIOD_NAME = f"from_2025_01_01_to_{DOWNLOAD_DATE}"
RUN_ID_FULL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

BATCH_SIZE = 50000

WRITE_PARQUET = True
WRITE_CSV = True

print("=" * 100)
print("SECOP BIG DATA - FINAL LAYER 0 DOWNLOAD")
print("=" * 100)
print("Start date:", START_DATE)
print("End date:", END_DATE)
print("Period:", PERIOD_NAME)
print("Run ID:", RUN_ID_FULL)
print("Base path:", BASE_PATH)
print("Batch size:", BATCH_SIZE)

# ============================================================
# 3. DATA SOURCES
# ============================================================

SOURCES_FULL = {
    "secop_ii_contratos": {
        "url": "https://www.datos.gov.co/resource/jbjy-vk9h.json",
        "date_column": "fecha_de_firma",
        "apply_date_filter": True
    },
    "secop_ii_adiciones": {
        "url": "https://www.datos.gov.co/resource/cb9c-h8sn.json",
        "date_column": "fecharegistro",
        "apply_date_filter": True
    },
    "secop_ii_ejecucion": {
        "url": "https://www.datos.gov.co/resource/mfmm-jqmq.json",
        "date_column": "fechacreacion",
        "apply_date_filter": True
    },
    "divipola_municipios": {
        "url": "https://www.datos.gov.co/resource/gdxc-w37w.json",
        "date_column": None,
        "apply_date_filter": False
    }
}

sources_to_download_full = list(SOURCES_FULL.keys())

# ============================================================
# 4. CREATE FOLDER STRUCTURE
# ============================================================

base_folders = [
    f"{BASE_PATH}/raw_jsonl",
    f"{BASE_PATH}/raw_csv",
    f"{BASE_PATH}/parquet",
    f"{BASE_PATH}/manifest",
    f"{BASE_PATH}/logs"
]

for folder in base_folders:
    os.makedirs(folder, exist_ok=True)

for source_name in sources_to_download_full:
    os.makedirs(f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}", exist_ok=True)
    os.makedirs(f"{BASE_PATH}/raw_csv/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}", exist_ok=True)
    os.makedirs(f"{BASE_PATH}/parquet/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}", exist_ok=True)

print("\nFolder structure created.")

# ============================================================
# 5. API PARAMETER BUILDER
# ============================================================

def build_params(source_config, limit, offset):
    params = {
        "$limit": limit,
        "$offset": offset
    }

    if source_config["apply_date_filter"]:
        date_column = source_config["date_column"]

        params["$where"] = (
            f"{date_column} >= '{START_DATE}' "
            f"AND {date_column} <= '{END_DATE}'"
        )

        params["$order"] = f"{date_column} ASC"

    return params

# ============================================================
# 6. API REQUEST WITH RETRIES
# ============================================================

def fetch_batch(source_config, limit, offset, retries=3, sleep_seconds=15):
    params = build_params(source_config, limit, offset)

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(
                source_config["url"],
                params=params,
                timeout=240
            )

            response.raise_for_status()
            data = response.json()

            if isinstance(data, dict) and "error" in data:
                raise Exception(data)

            return data

        except Exception as error:
            print(f"Attempt {attempt} failed | offset={offset}")
            print("Error:", error)

            if attempt == retries:
                raise

            print(f"Waiting {sleep_seconds} seconds before retry...")
            time.sleep(sleep_seconds)

# ============================================================
# 7. DOWNLOAD COMPLETE SOURCE
# ============================================================

def download_all_source(source_name):
    source_config = SOURCES_FULL[source_name]

    raw_jsonl_path = (
        f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"
    )

    os.makedirs(raw_jsonl_path, exist_ok=True)

    manifest = []
    total_records = 0
    batch_number = 0

    while True:
        offset = batch_number * BATCH_SIZE

        print("\n" + "=" * 100)
        print(f"Downloading source: {source_name}")
        print(f"Batch: {batch_number}")
        print(f"Offset: {offset}")
        print(f"Limit: {BATCH_SIZE}")
        print(f"Query window: {START_DATE} to {END_DATE}")

        records = fetch_batch(
            source_config=source_config,
            limit=BATCH_SIZE,
            offset=offset
        )

        received = len(records)
        total_records += received

        print(f"Records received: {received}")

        if received == 0:
            print("No more records available.")
            break

        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

        jsonl_file = (
            f"{raw_jsonl_path}/"
            f"batch_{batch_number:04d}_"
            f"offset_{offset}_"
            f"limit_{BATCH_SIZE}_"
            f"{timestamp}.jsonl"
        )

        with open(jsonl_file, "w", encoding="utf-8") as f:
            for record in records:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        manifest.append({
            "source": source_name,
            "period": PERIOD_NAME,
            "start_date": START_DATE,
            "download_end_date": END_DATE,
            "run_id": RUN_ID_FULL,
            "batch_number": batch_number,
            "offset": offset,
            "limit": BATCH_SIZE,
            "records": received,
            "jsonl_file": jsonl_file,
            "download_utc": datetime.now(timezone.utc).isoformat()
        })

        print(f"Saved JSONL: {jsonl_file}")

        if received < BATCH_SIZE:
            print("Last batch reached because received records are lower than batch size.")
            break

        batch_number += 1

        # Small pause to reduce API pressure
        time.sleep(3)

    manifest_file = (
        f"{BASE_PATH}/manifest/"
        f"manifest_{source_name}_{PERIOD_NAME}_run_{RUN_ID_FULL}.json"
    )

    with open(manifest_file, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=4)

    print("\n" + "=" * 100)
    print(f"Finished source: {source_name}")
    print(f"Total records downloaded: {total_records}")
    print(f"Manifest saved: {manifest_file}")

    return {
        "source": source_name,
        "period": PERIOD_NAME,
        "start_date": START_DATE,
        "download_end_date": END_DATE,
        "total_records": total_records,
        "batches": len(manifest),
        "manifest_file": manifest_file,
        "run_id": RUN_ID_FULL
    }

# ============================================================
# 8. DOWNLOAD ALL SOURCES
# ============================================================

full_download_summary = []

for source_name in sources_to_download_full:
    result = download_all_source(source_name)
    full_download_summary.append(result)

print("\n" + "=" * 100)
print("DOWNLOAD SUMMARY")
print("=" * 100)

for item in full_download_summary:
    print(item)

# ============================================================
# 9. VALIDATE RAW JSONL DOWNLOAD
# ============================================================

validation_summary = []

for source_name in sources_to_download_full:
    raw_jsonl_path = (
        f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"
    )

    print("\n" + "=" * 100)
    print(f"Validating raw JSONL source: {source_name}")
    print("Path:", raw_jsonl_path)

    try:
        files = dbutils.fs.ls(raw_jsonl_path)
        print(f"Files found: {len(files)}")

        if len(files) == 0:
            print("No files found.")
            row_count = 0
            col_count = 0
        else:
            df_raw = spark.read.json(raw_jsonl_path)
            row_count = df_raw.count()
            col_count = len(df_raw.columns)

            print(f"Rows: {row_count}")
            print(f"Columns: {col_count}")

        validation_summary.append({
            "source": source_name,
            "raw_jsonl_path": raw_jsonl_path,
            "files": len(files),
            "rows": row_count,
            "columns": col_count
        })

    except Exception as error:
        print(f"Validation error for {source_name}: {error}")
        validation_summary.append({
            "source": source_name,
            "raw_jsonl_path": raw_jsonl_path,
            "files": 0,
            "rows": 0,
            "columns": 0,
            "error": str(error)
        })

# ============================================================
# 10. CONVERT TO PARQUET AND CSV
# ============================================================

for source_name in sources_to_download_full:
    raw_jsonl_path = (
        f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"
    )

    parquet_path = (
        f"{BASE_PATH}/parquet/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"
    )

    csv_path = (
        f"{BASE_PATH}/raw_csv/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"
    )

    print("\n" + "=" * 100)
    print(f"Converting source: {source_name}")
    print("Raw JSONL path:", raw_jsonl_path)

    try:
        df = spark.read.json(raw_jsonl_path)

        rows = df.count()
        cols = len(df.columns)

        print(f"Rows to convert: {rows}")
        print(f"Columns to convert: {cols}")

        if rows == 0:
            print("Skipping conversion because there are no records.")
            continue

        if WRITE_PARQUET:
            (
                df.write
                .mode("overwrite")
                .parquet(parquet_path)
            )
            print(f"Parquet saved: {parquet_path}")

        if WRITE_CSV:
            (
                df.write
                .mode("overwrite")
                .option("header", "true")
                .csv(csv_path)
            )
            print(f"CSV saved: {csv_path}")

    except Exception as error:
        print(f"Conversion error for {source_name}: {error}")

# ============================================================
# 11. SAVE GLOBAL EXECUTION SUMMARY
# ============================================================

global_summary = {
    "project": "SECOP Big Data 2025",
    "layer": "Layer 0",
    "period": PERIOD_NAME,
    "start_date": START_DATE,
    "download_end_date": END_DATE,
    "run_id": RUN_ID_FULL,
    "batch_size": BATCH_SIZE,
    "sources": full_download_summary,
    "validation": validation_summary,
    "created_utc": datetime.now(timezone.utc).isoformat()
}

global_summary_file = (
    f"{BASE_PATH}/manifest/"
    f"global_summary_{PERIOD_NAME}_run_{RUN_ID_FULL}.json"
)

with open(global_summary_file, "w", encoding="utf-8") as f:
    json.dump(global_summary, f, ensure_ascii=False, indent=4)

print("\n" + "=" * 100)
print("GLOBAL SUMMARY SAVED")
print("=" * 100)
print(global_summary_file)

# ============================================================
# 12. FINAL DISPLAY
# ============================================================

print("\n" + "=" * 100)
print("FINAL RESULT")
print("=" * 100)
print("Period:", PERIOD_NAME)
print("Run ID:", RUN_ID_FULL)
print("Base path:", BASE_PATH)
print("Raw JSONL:", f"{BASE_PATH}/raw_jsonl")
print("Parquet:", f"{BASE_PATH}/parquet")
print("CSV:", f"{BASE_PATH}/raw_csv")
print("Manifest:", f"{BASE_PATH}/manifest")

display(full_download_summary)
display(validation_summary)

## Paso 33. Validar si una ejecución específica tiene datos

Esta celda revisa si existen archivos en una ruta específica usando `PERIOD_NAME` y `RUN_ID_FULL`.

Es útil cuando el proceso falla o cuando se necesita confirmar si una descarga quedó guardada antes de continuar.

In [0]:
BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

PERIOD_NAME = "from_2025_01_01_to_2026-06-02"  # replace if different
RUN_ID_FULL = "20260603T021642Z"               # replace with your real run_id

sources_to_download_full = [
    "secop_ii_contratos",
    "secop_ii_adiciones",
    "secop_ii_ejecucion",
    "divipola_municipios"
]

for source_name in sources_to_download_full:
    path = f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"

    print("=" * 80)
    print("SOURCE:", source_name)
    print("PATH:", path)

    try:
        files = dbutils.fs.ls(path)
        print("Files found:", len(files))
        display(files)

        if len(files) > 0:
            df = spark.read.json(path)
            print("Rows:", df.count())
            print("Columns:", len(df.columns))

    except Exception as e:
        print("No data found or error:", e)

## Paso 34. Buscar descargas reales en raw_jsonl

Este bloque explora automáticamente las carpetas dentro de `raw_jsonl`.

Su propósito es encontrar qué períodos y qué `run_id` realmente tienen archivos descargados.

Esto fue importante en el proyecto porque algunas ejecuciones generaron `run_id` diferentes o rutas incompletas. Esta celda ayuda a ubicar la descarga real.

In [0]:
BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

sources = [
    "secop_ii_contratos",
    "secop_ii_adiciones",
    "secop_ii_ejecucion",
    "divipola_municipios"
]

def safe_ls(path):
    try:
        return dbutils.fs.ls(path)
    except Exception as e:
        print(f"No existe o no se puede leer: {path}")
        return []

print("=" * 100)
print("BUSCANDO DATOS DESCARGADOS EN raw_jsonl")
print("=" * 100)

for source in sources:
    source_root = f"{BASE_PATH}/raw_jsonl/{source}"
    print("\n" + "=" * 100)
    print("SOURCE:", source)
    print("ROOT:", source_root)

    level_1 = safe_ls(source_root)

    for item_1 in level_1:
        print("  ", item_1.path)

        level_2 = safe_ls(item_1.path)

        for item_2 in level_2:
            print("      ", item_2.path)

            level_3 = safe_ls(item_2.path)

            jsonl_files = [
                f for f in level_3
                if f.path.endswith(".jsonl")
            ]

            if jsonl_files:
                print("          JSONL files found:", len(jsonl_files))
                for file in jsonl_files[:5]:
                    print("          -", file.name)

## Paso 35. Validar conteos del run correcto

Una vez identificado el `run_id` real, esta celda valida:

```text
número de archivos JSONL
número de filas
número de columnas
```

por cada fuente.

Esta validación confirma que la descarga fue exitosa antes de pasar a Parquet, Silver o Gold.

In [0]:
BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

PERIOD_NAME = "from_2025_01_01_to_2026-06-02"
RUN_ID_FULL = "20260603T031107Z"

sources_to_validate = [
    "secop_ii_contratos",
    "secop_ii_adiciones",
    "secop_ii_ejecucion",
    "divipola_municipios"
]

validation_results = []

for source_name in sources_to_validate:
    path = f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"

    print("=" * 100)
    print("SOURCE:", source_name)
    print("PATH:", path)

    try:
        files = dbutils.fs.ls(path)
        jsonl_files = [f for f in files if f.path.endswith(".jsonl")]

        print("JSONL files:", len(jsonl_files))

        if len(jsonl_files) > 0:
            df = spark.read.json(path)

            rows = df.count()
            columns = len(df.columns)

            print("Rows:", rows)
            print("Columns:", columns)

            validation_results.append({
                "source": source_name,
                "path": path,
                "jsonl_files": len(jsonl_files),
                "rows": rows,
                "columns": columns
            })
        else:
            print("No JSONL files found.")

            validation_results.append({
                "source": source_name,
                "path": path,
                "jsonl_files": 0,
                "rows": 0,
                "columns": 0
            })

    except Exception as e:
        print("No data found or error:", e)

        validation_results.append({
            "source": source_name,
            "path": path,
            "jsonl_files": 0,
            "rows": 0,
            "columns": 0,
            "error": str(e)
        })

display(validation_results)

## Paso 36. Reconstruir Parquet desde JSONL validado

Este paso toma los archivos JSONL confirmados y los convierte nuevamente a Parquet.

Esto corrige problemas donde existían carpetas Parquet vacías o incompletas.

La lógica correcta es:

```text
raw_jsonl validado → spark.read.json() → parquet limpio y usable
```

Con esto se deja lista la entrada para la fase Silver.

In [0]:
for item in validation_results:
    source_name = item["source"]

    if item["rows"] > 0:
        raw_path = item["path"]
        parquet_path = f"{BASE_PATH}/parquet/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"

        df = spark.read.json(raw_path)

        (
            df.write
            .mode("overwrite")
            .parquet(parquet_path)
        )

        print(f"Parquet saved for {source_name}: {parquet_path}")